<a href="https://colab.research.google.com/github/joeycocianci/L4-Data-Analytics_S2_Python-Spotify-Analysis/blob/main/spotify_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Spotify Top 50 2020: Exploratory Data Analysis
Author: Joe Bennett

Date: April 2026

Purpose: To analyze the musical features and artist trends of the 50 most-streamed tracks of 2020. This analysis focuses on data cleaning, statistical distribution, and artist popularity.

### Data Dictionary
| Feature | Type | Description |
| :--- | :--- | :--- |
| **Artist** | Categorical | The name of the performing artist. |
| **Track Name** | Categorical | The title of the song. |
| **Popularity** | Numeric | A score from 0-100 based on streams. |
| **Danceability** | Numeric | How suitable a track is for dancing (0.0 to 1.0). |
| **Energy** | Numeric | Perceptual measure of intensity and activity. |


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from io import StringIO

# This ensures your charts show up directly in the notebook
%matplotlib inline

# Phase 1: Data Loading with Error Handling
url = 'https://raw.githubusercontent.com/joeycocianci/L4-Data-Analytics_S2_Python-Spotify-Analysis/main/spotifytoptracks.csv'

try:
    # Try loading from GitHub first
    df = pd.read_csv(url)
    print("[SUCCESS] Dataset loaded successfully from GitHub!")
    display(df.head(5)) # Explicitly display the table

except Exception as e:
    print(f"[ERROR] GitHub link failed: {e}")
    print("Attempting to load from local session storage...")

    try:
        # Fallback to local file
        df = pd.read_csv('spotifytoptracks.csv')
        print("[SUCCESS] Dataset loaded successfully from local file!")
        display(df.head(5)) # Explicitly display the table
    except Exception as e_local:
        print(f"[ERROR] Local file failed too: {e_local}")
        df = None

# Final check to see if we have a valid DataFrame to work with
if df is not None:
    print(f"\nReady for analysis! Rows: {df.shape[0]}, Columns: {df.shape[1]}")
    print("\n--- Structural Audit ---")
    df.info()

[SUCCESS] Dataset loaded successfully from GitHub!


,Unnamed: 0,artist,album,track_name,track_id,energy,danceability,key,loudness,acousticness,speechiness,instrumentalness,liveness,valence,tempo,duration_ms,genre
0,0,The Weeknd,After Hours,Blinding Lights,0VjIjW4GlUZAMYd2vXMi3b,0.730,0.514,1,-5.934,0.00146,0.0598,0.000095,0.0897,0.334,171.005,200040,R&B/Soul
1,1,Tones And I,Dance Monkey,Dance Monkey,1rgnBhdG2JDFTbYkYRZAku,0.593,0.825,6,-6.401,0.68800,0.0988,0.000161,0.1700,0.540,98.078,209755,Alternative/Indie
2,2,Roddy Ricch,Please Excuse Me For Being Antisocial,The Box,0nbXyq5TXYPCO7pr3N8S4I,0.586,0.896,10,-6.687,0.10400,0.0559,0.000000,0.7900,0.642,116.971,196653,Hip-Hop/Rap
3,3,SAINt JHN,Roses (Imanbek Remix),Roses - Imanbek Remix,2Wo6QQD1KMDWeFkkjLqwx5,0.721,0.785,8,-5.457,0.01490,0.0506,0.004320,0.2850,0.894,121.962,176219,Dance/Electronic
4,4,Dua Lipa,Future Nostalgia,Don't Start Now,3PfIrDoz19wz7qK7tYeu62,0.793,0.793,11,-4.521,0.01230,0.0830,0.000000,0.0951,0.679,123.950,183290,Nu-disco



Ready for analysis! Rows: 50, Columns: 17

--- Structural Audit ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Unnamed: 0        50 non-null     int64  
 1   artist            50 non-null     object 
 2   album             50 non-null     object 
 3   track_name        50 non-null     object 
 4   track_id          50 non-null     object 
 5   energy            50 non-null     float64
 6   danceability      50 non-null     float64
 7   key               50 non-null     int64  
 8   loudness          50 non-null     float64
 9   acousticness      50 non-null     float64
 10  speechiness       50 non-null     float64
 11  instrumentalness  50 non-null     float64
 12  liveness          50 non-null     float64
 13  valence           50 non-null     float64
 14  tempo             50 non-null     float64
 15  duration_ms       50 non

In [2]:
# --- PHASE 2: CLEANING AND VERIFICATION (INTELLIGENT VERSION) ---

# 1. Capture counts for the report
initial_col_count = len(df.columns)
cols_to_drop = [c for c in df.columns if c.startswith('Unnamed:')]
num_cols_removed = len(cols_to_drop)

# 2. Perform the cleaning
# We use errors='ignore' to ensure the code remains idempotent
df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
df.drop_duplicates(inplace=True)

# 3. Calculate Data Integrity (Question 3 Improvement)
total_nulls = df.isnull().sum().sum()

# --- THE PROFESSIONAL INTEGRITY REPORT ---
print("="*40)
print("DATA INTEGRITY REPORT")
print("="*40)

# Addressing Idempotency
if num_cols_removed > 0:
    print(f"[ACTION] Cleaned {num_cols_removed} redundant index column(s).")
else:
    print("[STATUS] No redundant columns found. (Data is already lean)")

print(f"[STATUS] Final Data Shape: {len(df)} rows x {len(df.columns)} columns")
print("-" * 45)

# Logic: Only show the breakdown if there is an actual issue
if total_nulls > 0:
    print(f"ALERT: {total_nulls} Missing values detected!")
    print("\nBREAKDOWN BY COLUMN:")
    # Only displays columns where the null count is greater than zero
    print(df.isnull().sum()[df.isnull().sum() > 0])
else:
    print("NULL CHECK: No missing values detected in any column.")

print("-" * 45)
print(f"DUPLICATE CHECK: {df.duplicated().sum()} duplicates found.")
print("="*45)
print("RESULT: Data is verified and ready for Phase 3. ")
print("="*45)

DATA INTEGRITY REPORT
[ACTION] Cleaned 1 redundant index column(s).
[STATUS] Final Data Shape: 50 rows x 16 columns
---------------------------------------------
NULL CHECK: No missing values detected in any column.
---------------------------------------------
DUPLICATE CHECK: 0 duplicates found.
RESULT: Data is verified and ready for Phase 3. 


In [3]:
# --- PHASE 3: EXPLORATORY DATA ANALYSIS (EDA) ---

# --- INSIGHT BLOCK 1: DATASET STRUCTURE ---

# 1. Observations and Features
rows, cols = df.shape
print(f"There are {rows} observations (tracks) and {cols} features in this dataset.")

# 2. Identifying Feature Types
numeric_features = df.select_dtypes(include=['number']).columns.tolist()
categorical_features = df.select_dtypes(include=['object']).columns.tolist()

print(f"\nNumeric Features ({len(numeric_features)}):")
print(numeric_features)

print(f"\nCategorical Features ({len(categorical_features)}):")
print(categorical_features)

There are 50 observations (tracks) and 16 features in this dataset.

Numeric Features (11):
['energy', 'danceability', 'key', 'loudness', 'acousticness', 'speechiness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms']

Categorical Features (5):
['artist', 'album', 'track_name', 'track_id', 'genre']


In [11]:
# --- PHASE 3: EXPLORATORY DATA ANALYSIS (EDA) ---

# --- INSIGHT BLOCK 2: MARKET DOMINANCE ---

# 1. Artist Analysis
artist_counts = df['artist'].value_counts()
max_val_artist = artist_counts.max() # Renamed slightly to keep it distinct
tied_artists = artist_counts[artist_counts == max_val_artist]
multiple_track_artists = artist_counts[artist_counts > 1]
total_artists = df['artist'].nunique()

print(f"Total unique artists: {total_artists}")
print(f"Most popular artist: {artist_counts.idxmax()} ({max_val_artist} tracks)")
if len(tied_artists) > 1:
    print(f"Note: This is a tie between {len(tied_artists)} artists. "
          f"{artist_counts.idxmax()} is listed first because they appear higher in the Top 50 chart rank.")

print("\nArtists with more than 1 popular track:")
print(multiple_track_artists)

# --- DESIGNER'S NOTE ---
# In a technical audit, I keep the 'dtype' output to verify data integrity.
# If presenting to a business stakeholder, I would use .to_string() or a
# visualization (like a Bar Chart) to remove technical metadata for clarity.

print("-" * 30)

# 2. Album Analysis
album_counts = df['album'].value_counts()
multiple_track_albums = album_counts[album_counts > 1]
total_albums = df['album'].nunique()

print(f"Total unique albums: {total_albums}")
# We add a small check here too: if there are none, it tells us!
if multiple_track_albums.empty:
    print("\nNo albums have more than 1 popular track.")
else:
    print("\nAlbums with more than 1 popular track:")
    print(multiple_track_albums)

print("-" * 30)

# 3. Genre Analysis
genre_counts = df['genre'].value_counts()
single_song_genres = genre_counts[genre_counts == 1]
max_val_genre = genre_counts.max()
tied_genres = genre_counts[genre_counts == max_val_genre]

print(f"Total unique genres: {len(genre_counts)}")
print(f"Most popular genre: {genre_counts.idxmax()} ({max_val_genre} tracks)")

if len(tied_genres) > 1:
    # Fixed the 'idmax' to 'idxmax' here!
    print(f"Note: This is a tie between {len(tied_genres)} genres. "
          f"{genre_counts.idxmax()} is listed first because the first song of that genre appears higher in the Top 50 chart rank.")

print(f"\nGenres with only one song in the top 50: {len(single_song_genres)}")
print(single_song_genres.index.tolist())

Total unique artists: 40
Most popular artist: Dua Lipa (3 tracks)
Note: This is a tie between 3 artists. Dua Lipa is listed first because they appear higher in the Top 50 chart rank.

Artists with more than 1 popular track:
artist
Dua Lipa         3
Billie Eilish    3
Travis Scott     3
Harry Styles     2
Lewis Capaldi    2
Justin Bieber    2
Post Malone      2
Name: count, dtype: int64
------------------------------
Total unique albums: 45

Albums with more than 1 popular track:
album
Future Nostalgia        3
Hollywood's Bleeding    2
Fine Line               2
Changes                 2
Name: count, dtype: int64
------------------------------
Total unique genres: 16
Most popular genre: Pop (14 tracks)

Genres with only one song in the top 50: 10
['R&B/Hip-Hop alternative', 'Nu-disco', 'Pop/Soft Rock', 'Pop rap', 'Hip-Hop/Trap', 'Dance-pop/Disco', 'Disco-pop', 'Dreampop/Hip-Hop/R&B', 'Alternative/reggaeton/experimental', 'Chamber pop']
